# Week 2 — Problematizing Data and Algorithmic Bias

* * *

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Understand what the COMPAS recidivism dataset is, where it comes from, and why ProPublica's investigation made it a touchstone for algorithmic-bias debates.
* Practice core `pandas` operations: loading, inspecting, filtering, grouping, and cross-tabulating.
* See — concretely, in code — how data cleaning is *never* a neutral operation.
* Reproduce a simplified version of ProPublica's finding: that COMPAS's false-positive rate is much higher for Black defendants than for White defendants.
* Connect the technical findings to Hoffmann's argument that "fairness" reforms can miss the structural problem.
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.

### Sections
1. [Framing: Where Fairness Fails](#framing)
2. [What Is COMPAS?](#compas)
3. [Loading the Data](#load)
4. [Pandas Basics: Looking at the Categories We're Given](#basics)
5. [Data Cleaning Is Never Neutral](#cleaning)
6. [Exploring Risk Scores by Race](#explore)
7. [Reproducing the ProPublica Finding](#propublica)
8. [Reading: Hoffmann, *Where Fairness Fails*](#hoffmann)
9. [Looking Ahead: Bolukbasi et al. and Linguistic Bias](#ahead)
10. [Reflection Prompts](#reflection)

<a id='framing'></a>
# 1. Framing: Where Fairness Fails

Last week we asked: *who gets counted, and who is rendered invisible?* This week we put that question to one specific dataset that has shaped national debates about race and algorithms in the United States: **COMPAS**, a recidivism-risk score used by courts.

Monday's reading is Anna Lauren Hoffmann's 2019 essay, *Where Fairness Fails: Data, Algorithms, and the Limits of Antidiscrimination Discourse*. Her core move is unsettling: she argues that the dominant push for "algorithmic fairness" — typically framed as making sure outcomes are equal across protected groups — *imports the limits of antidiscrimination law* into computer science. And antidiscrimination law, she argues, has never been very good at addressing structural injustice. It's better at flagging *individual* unequal treatment than at confronting why the baseline is what it is.

Keep this critique in your back pocket as we dig into the data.

<a id='compas'></a>
# 2. What Is COMPAS?

**COMPAS** = *Correctional Offender Management Profiling for Alternative Sanctions*. It's a proprietary risk-assessment tool produced by the company Northpointe (now Equivant), used in many US courts to predict whether a defendant will be re-arrested within two years. Judges have used those predictions to inform decisions about bail, sentencing, and parole.

In 2016, journalists at **ProPublica** obtained risk scores for over 7,000 people arrested in Broward County, Florida, and compared the predictions to what actually happened. Their headline finding: COMPAS labeled Black defendants as future criminals at nearly *twice the rate* of White defendants who didn't go on to reoffend. White defendants, conversely, were more often labeled "low risk" and *did* go on to reoffend at higher rates than the label predicted. The exposé, *Machine Bias* (May 2016), is one of the most-cited pieces of data journalism of the last decade.

Northpointe contested the analysis. A long, technical debate followed. Several of those points (about which fairness metric is the right one) we'll see in Week 3. For today, the *dataset* itself is our subject.

> **Data transparency note**: The COMPAS data we'll use was obtained by ProPublica via Florida public records requests and posted on GitHub. It is *real* data about real people. Several caveats matter before we touch it:
>
> - The target column, `two_year_recid`, records whether someone was *re-arrested* within two years — not whether they committed a crime. Arrest is a measure of police contact, which is uneven across neighborhoods and races.
> - The `race` column is binary-coded (Black/White) for the bulk of the analysis. Other groups exist in the data but at much smaller numbers; ProPublica's analysis (and our reproduction) compares two groups, which itself reproduces a binary framing of US racial categorization.
> - These are *real names and outcomes*. We treat the data with care: we don't read individual rows for entertainment, and we don't share derived results outside this course context.
> - The data reflects one county (Broward, FL) at one period of time. Patterns may differ elsewhere — but the same kinds of questions apply everywhere.

<a id='load'></a>
# 3. Loading the Data

In [ ]:
#%pip install pandas matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
df.head()

In [ ]:
# What columns do we have?
list(df.columns)

💡 **Tip**: When you first load a dataset, always look at `df.shape`, `df.head()`, and `df.columns`. These three calls answer: *how big*, *what does a row look like*, and *what attributes were measured*. Then ask yourself the harder fourth question: *what wasn't measured?*

<a id='basics'></a>
# 4. Pandas Basics: Looking at the Categories We're Given

Let's see what categories the dataset offers us — and, just as importantly, what categories it doesn't.

In [ ]:
df["race"].value_counts()

In [ ]:
df["sex"].value_counts()

In [ ]:
df["c_charge_degree"].value_counts()    # F = felony, M = misdemeanor

💭 **Reflection**: The `race` column has six categories, but "African-American" and "Caucasian" overwhelmingly dominate the count. Smaller groups (Native American, Asian) appear in the dozens. 

The `sex` column has two values. There is no representation for non-binary, intersex, or trans defendants — even though such defendants certainly exist in the underlying population.

What does the dataset's *category structure* (binary sex; numerically dominant Black/White race) tell us about whose experiences are made legible to the analysis, and whose are erased?

<a id='cleaning'></a>
# 5. Data Cleaning Is Never Neutral

"Data cleaning" sounds neutral and technical. It isn't. Every cleaning step is a decision about which rows to count and which to drop.

Let's look at missingness:

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

Now let's narrow the data to the analytic subset that ProPublica used. **Each step below is a political choice as much as a technical one.** I'll annotate each.

In [ ]:
# Step 1: keep only Black and White defendants
# WHAT THIS EXCLUDES: Native American, Asian, Hispanic, and "Other" defendants.
# We are reproducing the racial binary that organized the original ProPublica analysis.
# Naming this is part of doing the analysis honestly.
df_clean = df[df["race"].isin(["African-American", "Caucasian"])].copy()
print("After step 1:", df_clean.shape)

In [ ]:
# Step 2: rename for readability
# WHAT THIS EXCLUDES: nothing data-wise, but renaming "African-American" to "Black"
# and "Caucasian" to "White" is also a political choice — it adopts the language
# many activists and scholars prefer over the older Census labels.
df_clean["race"] = df_clean["race"].replace({"African-American": "Black", "Caucasian": "White"})
df_clean["race"].value_counts()

In [ ]:
# Step 3: drop rows where the score or recidivism outcome is missing
# WHAT THIS EXCLUDES: defendants whose case files were incomplete.
# Missingness is rarely random; people with chaotic lives often have chaotic records.
df_clean = df_clean.dropna(subset=["decile_score", "two_year_recid"])
print("After step 3:", df_clean.shape)

In [ ]:
# Step 4: filter to charges that occurred within 30 days of arrest
# (matches the original ProPublica analysis; excludes likely data-entry errors)
# WHAT THIS EXCLUDES: about 8% of the data; mostly cases that look like
# administrative errors but a few that may be unusual real cases.
df_clean = df_clean[(df_clean["days_b_screening_arrest"] <= 30) & (df_clean["days_b_screening_arrest"] >= -30)]
print("After step 4:", df_clean.shape)

⚠️ **Warning**: Real-world data analysis pipelines often present "cleaning" as a single step in a chart. In a paper or a tutorial you might see the words "after standard preprocessing" — and that phrase can hide a dozen value-laden choices. When you read someone else's analysis, look for the cleaning steps. When you write your own, *show your work*.

<a id='explore'></a>
# 6. Exploring Risk Scores by Race

The `decile_score` is COMPAS's risk score, integer from 1 (lowest) to 10 (highest). Let's see how it's distributed by race.

In [ ]:
df_clean.groupby("race")["decile_score"].describe()

In [ ]:
# Recidivism rate by race (the *actual* outcome, not the prediction)
df_clean.groupby("race")["two_year_recid"].mean().round(3)

In [ ]:
# How does COMPAS bin people into Low / Medium / High?
pd.crosstab(df_clean["race"], df_clean["score_text"], normalize="index").round(3)

In [ ]:
# Visualize: histograms of risk score by race, overlaid
fig, ax = plt.subplots(figsize=(9, 5))
for race, color in [("Black", "steelblue"), ("White", "tomato")]:
    subset = df_clean[df_clean["race"] == race]["decile_score"]
    ax.hist(subset, bins=range(1, 12), alpha=0.5, label=race, color=color)
ax.set_xlabel("COMPAS decile score (1 = lowest risk, 10 = highest)")
ax.set_ylabel("Number of defendants")
ax.set_title("Risk-score distribution by race")
ax.legend()
plt.tight_layout()
plt.show()

🔔 **Question**: White defendants pile up at the low end of the risk scale. Black defendants are distributed much more evenly across all 10 deciles. What does this picture *show* us — and what does it *not yet* tell us?

(It does not, by itself, prove bias. The two distributions could differ for many reasons — including underlying differences in arrest histories, which themselves reflect uneven policing. The rest of this section makes the bias question more precise.)

<a id='propublica'></a>
# 7. Reproducing the ProPublica Finding

ProPublica's central claim was about **error rates**. They asked: among defendants who *did not* recidivate, how often did COMPAS predict that they would? That's the **false positive rate (FPR)**.

Let's compute it ourselves. We'll define "COMPAS predicted high-risk" as `decile_score >= 5` (medium or high) — matching ProPublica's threshold.

In [ ]:
df_clean["predicted_high_risk"] = (df_clean["decile_score"] >= 5).astype(int)

# Among those who did NOT reoffend, what fraction were labeled high risk?
non_recid = df_clean[df_clean["two_year_recid"] == 0]
fpr_by_race = non_recid.groupby("race")["predicted_high_risk"].mean().round(3)
print("False Positive Rate by race (lower is better):")
print(fpr_by_race)

In [ ]:
# And the false NEGATIVE rate: among those who DID reoffend, how often
# were they labeled low-risk?
recid = df_clean[df_clean["two_year_recid"] == 1]
fnr_by_race = (1 - recid.groupby("race")["predicted_high_risk"].mean()).round(3)
print("False Negative Rate by race (lower is better):")
print(fnr_by_race)

💭 **Reflection**: Read the numbers carefully. Black defendants who *did not* reoffend were labeled high-risk at a much higher rate than White defendants who didn't reoffend (false positives). White defendants who *did* reoffend were labeled low-risk at a higher rate than Black defendants who reoffended (false negatives).

In human terms: this means the system is *more likely* to recommend harsher treatment for a Black person who poses no future risk, *and* more likely to recommend leniency for a White person who does. The error pattern is not symmetric — and the asymmetry tracks race.

This is one technical definition of "unfair." In Week 3 we'll see why it isn't the *only* definition, and why the choice between definitions is itself a values question — not a math question.

<a id='hoffmann'></a>
# 8. Reading: Hoffmann, *Where Fairness Fails*

Now loop back to the Hoffmann reading. Her argument matters here in a very specific way:

> Even if Northpointe "fixed" COMPAS so that error rates were equal across races, the *underlying input* — arrests as a proxy for criminality — would still encode the unevenness of policing. The model would no longer *display* the disparity; it would just relocate it.

Hoffmann calls this the limit of antidiscrimination thinking: it is good at policing the *most* visible forms of unequal treatment, but it tends to leave the structures that produce uneven baselines untouched. A truly just response to a system like COMPAS may not be "a less biased COMPAS" but rather "do we want this kind of decision to be automated at all?"

Hold that thought for the rest of the term — we'll see versions of it in geospatial analysis (Week 4) and in language models (Week 5).

<a id='ahead'></a>
# 9. Looking Ahead: Bolukbasi et al. and Linguistic Bias

Tuesday's reading is Bolukbasi et al. (2016), *Man is to Computer Programmer as Woman is to Homemaker?* Same pattern as COMPAS, different domain. They show that word embeddings learned from large text corpora encode gender stereotypes — and they propose a "debiasing" method to remove them.

🔔 **Question**: Before next week, predict: do you think their debiasing method works? What would success even look like? (We'll dig into this in Week 5.)

### Other tools you could use for this kind of analysis

We're using `pandas` because it's beginner-friendly and dominant in industry — but tabular work like this can be done in many other tools. We'll mention these briefly in class:

- **R + `dplyr` / `data.table`** — the workhorse of statistical and academic data analysis. The same group-by / filter / summarize logic, different syntax.
- **OpenRefine** — a free GUI tool excellent for the *cleaning* steps in §5 (deduplication, clustering near-duplicates, faceted filtering).
- **SQL** (via SQLite, DuckDB, BigQuery) — when datasets get large or live in a database.
- **Excel / Google Sheets** — fine for a quick first look at a small dataset; check out conditional formatting and pivot tables before reaching for code.

The critical questions of this notebook (what does each column actually measure? what does the cleaning exclude?) apply identically in any of these tools.

<a id='reflection'></a>
# 10. Reflection Prompts

For your 300-word reflection on algorithmic bias, you can start from any of:

1. Pick one cleaning step in Section 5 and write 100 words on what was excluded and why it matters. Connect to a principle from D'Ignazio & Klein.

2. Hoffmann argues that "fixing" algorithmic bias can leave the underlying inequity untouched. Apply her argument to COMPAS: imagine Northpointe equalized FPR/FNR across races tomorrow. What would still be unjust about the system?

3. Re-read the data transparency note in Section 2. What additional caveats would *you* want to add, after spending the last hour with this dataset?

4. The dataset includes `c_charge_degree` (felony / misdemeanor) and `priors_count`. Both look like "objective" attributes of a person. In what ways are they actually attributes of the *system that produced the record*?

<div class="alert alert-success">

## ❗ Key Points

* COMPAS is a recidivism-risk algorithm used in US courts; ProPublica's 2016 investigation found racial disparities in its error rates.
* The dataset's *categories* (binary sex; numerically dominant Black/White race) shape what the analysis can see.
* Every "data cleaning" step is also a choice about whose experiences to include and exclude.
* Black defendants who didn't reoffend were labeled high-risk at much higher rates than White defendants who didn't (higher false positive rate); the inverse held for false negatives.
* The model never used `race` directly — but features correlated with race (priors, charges) do the work of race anyway. We'll dig into this proxy-variable problem in Week 3.
* Hoffmann reminds us: making an algorithm "fair" by some metric doesn't address the structural unevenness in its inputs. Fairness fixes can hide injustice as easily as expose it.

</div>